# Piloto: contexto e questões por cobertura
Entrada: tópico, subtópico e frase integral do especialista. Padrão: **Produtos químicos (02_02)**.
Execute com o ambiente do servidor e o corpus profissional já instalado. As células de geração chamam Azure. O planejamento e a leitura de resultados não chamam modelos. Veja `README.md` para instalação, fontes e retomada.

In [ ]:
from pathlib import Path
import json, subprocess, sys, os
from IPython.display import Markdown, display
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pipeline/fase_3/topicos.py').exists())
os.chdir(root)
item = '02_02'
manifest = root / 'dataset/professional_corpora_v2/manifest.json'
output = root / 'pipeline/qa_research/saida_cobertura_v2' / item
base = [sys.executable, '-u', '-m', 'pipeline.qa_research.v2', '--item', item, '--manifest', str(manifest), '--output', str(output)]
print('Projeto:', root)
print('Python:', sys.executable)

## Conferir o plano e os modelos — sem chamadas
O título e a frase abaixo vêm do programa original. Nenhum recorte substitui o subtópico.

In [ ]:
subprocess.run(base + ['--plan'], check=True)

## Gerar o contexto — usa Azure
Planejamento → pesquisa no corpus → redação/auditoria → revisão global e novas buscas. Código 2 indica resultado parcial ou interrupção; consulte os relatórios antes de continuar.

In [ ]:
result = subprocess.run(base + ['--stage', 'context'])
print('Código de saída:', result.returncode)
if (output / 'RELATORIO.md').exists(): display(Markdown((output / 'RELATORIO.md').read_text(encoding='utf-8')))

In [ ]:
if (output / 'contexto/documento.md').exists():
    display(Markdown((output / 'contexto/documento.md').read_text(encoding='utf-8')))

## Gerar questões — usa Azure
Inventário auditado → alvos por classe → autoria → resolução sem gabarito → auditoria com contexto → refino encaminhado → deduplicação. A etapa bloqueia se persistirem conflitos na revisão global do documento.

In [ ]:
result = subprocess.run(base + ['--stage', 'qa'])
print('Código de saída:', result.returncode)
if (output / 'RELATORIO.md').exists(): display(Markdown((output / 'RELATORIO.md').read_text(encoding='utf-8')))

In [ ]:
report_path = output / 'questoes/cobertura.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps({k:v for k,v in report.items() if k != 'historico'}, ensure_ascii=False, indent=2))
print('Questões:', output / 'questoes/qa.csv')

Aprovação automática não é validação humana. O terminal imprime os comandos de coleta e envio pelo Git ao terminar cada etapa. A coleta exclui fontes brutas e caches; a auditoria completa permanece na saída do servidor. Reexecutar as mesmas células retoma os checkpoints.